# Описание ДЗ-3.

1. В третьей домашке необходимо для датасета животных обучить MLP.
2. Использовать Custom Dataset, Sampler, collate_fn
3. Сделать предобработку фичей
4. Попробовать BatchNorm1d, Dropout
5. Подключить для логирования tensorboard и/или mlflow
6. Не забыть разделить выборку на train/valid в соотношении 80/20%
7. Получить точность не ниже 65%.
8. Сравнить результаты при разных подходах: 1) Масштабировать данные StandardScaler'ом + не использовать BatchNorm после Input слоя 2) Не Масштабировать данные + использовать BatchNorm после Input слоя 3) StandardScaler + BatchNorm

*В папке LESS  можно найти примеры использования всех необходимых либ для выполнения данной работы*

За дз можно получить максимум 15 баллов. **Домашки довольно творческие, если замечу копию нотбука у другого студента то максимальный балл сниижается до 3 )**

Разбалловка:
*   **Воспроизводимость и читабельность кода -  11 баллов** (все воспроизвелось и все понятно для проверяющего - 11 баллов; есть непонятные моменты, но все воспроизвелось - 6 балла; непонятный код и/или воспроизводится с небольшой правкой - 3 балл; непонятный код и/или ничего не воспроизвелось - 0 баллов).
*   **Технический отчет - 4 балла** (приведены результаты сравнения и выводы что сделали чтоб перебить baseline\другую модель, к примеру одна модель лучше/хуже нейронки и тд - 4 балла, только результаты - 2 балл, ничего нет - 0 баллов).


**Присылать домашки по ссылке https://forms.gle/W8jwbwA4EWagEbX66**

In [213]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Sampler
from torch.utils.tensorboard import SummaryWriter
import torch.optim as optim

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

import random

Зафиксируем сид для воспроизведения результатов

In [214]:
SEED = 50

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


# Custom Dataset, Custom Sampler, collate_fn

In [215]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

    def __len__(self):
        return len(self.X)

Простой сэмплер: отдаёт все индексы в случайном порядке.

In [216]:
class CustomSampler(Sampler[int]):
    def __init__(self, num_samples: int):
        self.num_samples = num_samples

    def __iter__(self):
        indices = np.arange(self.num_samples)
        np.random.shuffle(indices)
        return iter(indices.tolist())

    def __len__(self):
        return self.num_samples

In [217]:
def collate_fn(batch):
    xs, ys = zip(*batch)
    return torch.tensor(xs), torch.tensor(ys)

# Функция предобработки фичей

Предобработка и отбор признаков на основе их значимости, вычисленной при помощи случайного леса.

In [218]:
def preprocess_data(X_df: pd.DataFrame,
                    y_df: pd.DataFrame,
                    n_top: int = 26,
                    n_estimators: int = 100,
                    random_state: int = SEED
                   ):
    X = X_df.values.astype(np.float32)
    y = y_df.values.squeeze()

    rf = RandomForestClassifier(
        n_estimators=n_estimators, 
        random_state=random_state
    )
    rf.fit(X, y)

    importances = rf.feature_importances_
    feature_importances_df = pd.DataFrame({
        'feature': X_df.columns,
        'importance': importances
    }).sort_values('importance', ascending=False).reset_index(drop=True)

    top_features = feature_importances_df['feature'].tolist()[:n_top]
    inds_selected = [i for i, col in enumerate(X_df.columns) 
                     if col in top_features]
    X_selected = X[:, inds_selected]

    return X_selected, y, feature_importances_df, top_features



# MLP класс

In [252]:
class MLP(nn.Module):
    def __init__(self, input_dim, use_bn=False, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 120)
        self.bn1 = nn.BatchNorm1d(120) if use_bn else nn.Identity()
        self.drop1 = nn.Dropout(dropout)
        self.fc2 = nn.Linear(120, 128)
        self.drop2 = nn.Dropout(dropout)
        self.fc3 = nn.Linear(128, 8)
        self.act = nn.ReLU()

    def forward(self, x):
        x = self.act(self.bn1(self.fc1(x)))
        x = self.drop1(x)
        x = self.act(self.fc2(x))
        x = self.drop2(x)
        return self.fc3(x)

# Функции для обучения

In [242]:
def run_epoch(model, loader, loss_fn, device, optimizer=None):
    is_train = optimizer is not None
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    preds, targets = [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if is_train:
                optimizer.zero_grad()
            out = model(x)
            loss = loss_fn(out, y)
            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            preds.append(out.argmax(dim=1).cpu())
            targets.append(y.cpu())

    preds = torch.cat(preds).numpy()
    targets = torch.cat(targets).numpy()
    return total_loss / len(loader), accuracy_score(targets, preds)


def train_epoch(model, loader, loss_fn, optimizer, device):
    return run_epoch(model, loader, loss_fn, device, optimizer=optimizer)


def evaluate(model, loader, loss_fn, device):
    return run_epoch(model, loader, loss_fn, device, optimizer=None)


def train(scaling, batchnorm, name, X, y, epochs=20, batch_size=64, lr=1e-3):
    # устройство и логгинг
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    writer = SummaryWriter(log_dir=f'logs/{name}')

    # скейлер
    X_proc = StandardScaler().fit_transform(X) if scaling else X

    # датасет и сплит
    dataset = CustomDataset(X_proc, y)
    n_train = int(0.8 * len(dataset))
    train_ds, valid_ds = random_split(dataset, [n_train, len(dataset) - n_train])

    # загрузчики
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size,
                              shuffle=False, collate_fn=collate_fn)

    # модель, оптимизатор, loss
    model = MLP(X.shape[1], use_bn=batchnorm, dropout=0.2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
        va_loss, va_acc = evaluate(model, valid_loader, loss_fn, device)

        writer.add_scalars("Loss",   {"train": tr_loss, "valid": va_loss}, epoch)
        writer.add_scalars("Accuracy", {"train": tr_acc, "valid": va_acc}, epoch)

        if epoch % 10 == 0 or epoch == epochs:
            print(f"{name}: Epoch {epoch} — "
                f"train_acc={tr_acc:.3f}, valid_acc={va_acc:.3f}")

    writer.close()
    return va_acc


# Dataset

In [221]:
X_df = pd.read_csv('Data-20250520T015757Z-1-001/X_cat.csv', sep='\t', index_col=[0])
y_df = pd.read_csv('Data-20250520T015757Z-1-001/y_cat.csv', sep='\t',
                names=['index', 'y'], header=None, index_col=[0])

In [222]:
X_df.head()

,IsDog,Age,HasName,NameLength,NameFreq,MixColor,ColorFreqAsIs,ColorFreqBase,TabbyColor,MixBreed,...,SexStatus_Flawed,SexStatus_Intact,SexStatus_Unknown,Weekday_0,Weekday_1,Weekday_2,Weekday_3,Weekday_4,Weekday_5,Weekday_6
0,1,365.0,1,7,0.000157,1,0.032919,0.463624,0,1,...,1,0,0,0,0,1,0,0,0,0
1,0,365.0,1,5,0.000655,0,0.008092,0.015005,1,1,...,1,0,0,0,0,0,0,0,0,1
2,1,730.0,1,6,0.000052,1,0.026293,0.357521,0,1,...,1,0,0,0,0,0,0,0,1,0
3,0,21.0,0,7,0.285871,0,0.000471,0.058418,0,1,...,0,1,0,0,0,0,0,1,0,0
4,1,730.0,0,7,0.285871,0,0.023831,0.075353,0,0,...,1,0,0,0,0,0,0,1,0,0


In [223]:
X_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26729 entries, 0 to 26728
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   IsDog                           26729 non-null  int64  
 1   Age                             26729 non-null  float64
 2   HasName                         26729 non-null  int64  
 3   NameLength                      26729 non-null  int64  
 4   NameFreq                        26729 non-null  float64
 5   MixColor                        26729 non-null  int64  
 6   ColorFreqAsIs                   26729 non-null  float64
 7   ColorFreqBase                   26729 non-null  float64
 8   TabbyColor                      26729 non-null  int64  
 9   MixBreed                        26729 non-null  int64  
 10  Domestic                        26729 non-null  int64  
 11  Shorthair                       26729 non-null  int64  
 12  Longhair                        26729

In [224]:
y_df.head()

,y
index,
0,Return_to_owner
1,Euthanasia
2,Adoption
3,Transfer
4,Transfer


In [225]:
y_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 26729 entries, 0 to 26728
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   y       26729 non-null  object
dtypes: object(1)
memory usage: 417.6+ KB


In [226]:
y_df['y'].unique()

array(['Return_to_owner', 'Euthanasia', 'Adoption', 'Transfer', 'Died'],
      dtype=object)

In [227]:
# Заменим категориальные переменные на числовые коды
codes, uniques = pd.factorize(y_df['y'])
y_df['y'] = codes + 1

In [228]:
y_df.head()

,y
index,
0,1
1,2
2,3
3,4
4,4


In [ ]:
X, y, _, _ = preprocess_data(X_df, y_df)

# Обучение моделей

In [253]:
default = train(False, False, "default", X, y, epochs=50)
print(f"Standart Scaler and BatchNorm: {default:.2%}")

default: Epoch 10 — train_acc=0.485, valid_acc=0.525
default: Epoch 20 — train_acc=0.593, valid_acc=0.624
default: Epoch 30 — train_acc=0.599, valid_acc=0.624
default: Epoch 40 — train_acc=0.590, valid_acc=0.619
default: Epoch 50 — train_acc=0.591, valid_acc=0.610
Standart Scaler and BatchNorm: 61.04%


In [260]:
mlp_standart_only = train(True, False,"standard_only", X, y, epochs=50)
print(f"StandardScaler only: {mlp_standart_only:.2%}")

standard_only: Epoch 10 — train_acc=0.668, valid_acc=0.667
standard_only: Epoch 20 — train_acc=0.674, valid_acc=0.669
standard_only: Epoch 30 — train_acc=0.685, valid_acc=0.668
standard_only: Epoch 40 — train_acc=0.690, valid_acc=0.669
standard_only: Epoch 50 — train_acc=0.692, valid_acc=0.665
StandardScaler only: 66.48%


In [255]:
mlp_batchnorm_only = train(False, True, "batchnorm_only", X, y, epochs=50)
print(f"BatchNorm only: {mlp_batchnorm_only:.2%}")

batchnorm_only: Epoch 10 — train_acc=0.623, valid_acc=0.376
batchnorm_only: Epoch 20 — train_acc=0.630, valid_acc=0.644
batchnorm_only: Epoch 30 — train_acc=0.637, valid_acc=0.577
batchnorm_only: Epoch 40 — train_acc=0.639, valid_acc=0.631
batchnorm_only: Epoch 50 — train_acc=0.645, valid_acc=0.652
BatchNorm only: 65.21%


In [258]:
mlm_both = train(True, True, "standard_and_batchnorm", X, y, epochs=50)
print(f"Standart Scaler and BatchNorm: {mlm_both:.2%}")

standard_and_batchnorm: Epoch 10 — train_acc=0.662, valid_acc=0.674
standard_and_batchnorm: Epoch 20 — train_acc=0.671, valid_acc=0.680
standard_and_batchnorm: Epoch 30 — train_acc=0.677, valid_acc=0.681
standard_and_batchnorm: Epoch 40 — train_acc=0.680, valid_acc=0.683
standard_and_batchnorm: Epoch 50 — train_acc=0.686, valid_acc=0.678
Standart Scaler and BatchNorm: 67.77%


MLP без StandardScaler и без BatchNorm: 61.04%

MLP с StandardScaler без BatchNorm: 66.48%.

MLP без StandardScaler с BatchNorm: 65.21%.

MLP с StandardScaler и BatchNorm: 67.77%.

Итог: Использование StandardScaler и BatchNorm по отдельности улучшает качество модели по сравнению с базовой MLP. Наилучший результат достигается при совместном применении StandardScaler и BatchNorm.